# HW08-09 — PyTorch MLP (S08+S09)

Датасет: EMNIST (balanced) (torchvision).

Практическая работа 8-9 - Атаманчук А.В. КВБО-01-22

In [13]:
import json
import math
import os
import random
import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook cwd:", PROJECT_DIR)
print("Artifacts dir:", ARTIFACTS_DIR)

Notebook cwd: d:\VUZ\II_DOP_KURS\MIREA_III\homeworks\HW08-09
Artifacts dir: d:\VUZ\II_DOP_KURS\MIREA_III\homeworks\HW08-09\artifacts


In [9]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


SEED = 42
set_seed(SEED)
DEVICE = get_device()

print("seed:", SEED)
print("device:", DEVICE)
print("torch:", torch.__version__)
try:
    import torchvision

    print("torchvision:", torchvision.__version__)
except Exception as e:
    print("torchvision import error:", e)

seed: 42
device: cuda
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128


In [10]:
# Data
BATCH_SIZE = 1024
VAL_RATIO = 0.2

transform = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize((0.5,), (0.5,)),  # optional
])

data_root = PROJECT_DIR / "data"

train_full = datasets.EMNIST(split="balanced", root=str(data_root), train=True, download=True, transform=transform)
test_ds = datasets.EMNIST(split="balanced", root=str(data_root), train=False, download=True, transform=transform)

val_len = int(len(train_full) * VAL_RATIO)
train_len = len(train_full) - val_len

# Reproducible split
split_gen = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(train_full, [train_len, val_len], generator=split_gen)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=(DEVICE.type == "cuda"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=(DEVICE.type == "cuda"))

# Sanity-check batch
x, y = next(iter(train_loader))
print("batch x:", x.shape, x.dtype, "min/max", float(x.min()), float(x.max()))
print("batch y:", y.shape, y.dtype, "labels", y[:10].tolist())

batch x: torch.Size([1024, 1, 28, 28]) torch.float32 min/max 0.0 1.0
batch y: torch.Size([1024]) torch.int64 labels [32, 3, 36, 29, 28, 23, 39, 2, 9, 10]


In [11]:
class MLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        hidden_sizes: List[int],
        dropout_p: float = 0.0,
        use_batchnorm: bool = False,
    ):
        super().__init__()

        layers: List[nn.Module] = [nn.Flatten()]
        prev = input_dim

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p and dropout_p > 0:
                layers.append(nn.Dropout(p=dropout_p))
            prev = h

        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        bs = yb.size(0)
        total_loss += loss.item() * bs
        total_correct += (torch.argmax(logits, dim=1) == yb).sum().item()
        total += bs

    return total_loss / total, total_correct / total


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        bs = yb.size(0)
        total_loss += loss.item() * bs
        total_correct += (torch.argmax(logits, dim=1) == yb).sum().item()
        total += bs

    return total_loss / total, total_correct / total

In [12]:
@dataclass
class EarlyStopping:
    patience: int = 5
    mode: str = "max"  # 'max' for accuracy, 'min' for loss
    min_delta: float = 0.0

    def __post_init__(self):
        if self.mode not in {"max", "min"}:
            raise ValueError("mode must be 'max' or 'min'")
        self.best_value: Optional[float] = None
        self.num_bad_epochs: int = 0
        self.should_stop: bool = False

    def step(self, value: float) -> bool:
        if self.best_value is None:
            self.best_value = value
            self.num_bad_epochs = 0
            return False

        improved = False
        if self.mode == "max":
            improved = value > (self.best_value + self.min_delta)
        else:
            improved = value < (self.best_value - self.min_delta)

        if improved:
            self.best_value = value
            self.num_bad_epochs = 0
        else:
            self.num_bad_epochs += 1
            if self.num_bad_epochs >= self.patience:
                self.should_stop = True

        return self.should_stop


def model_summary(hidden_sizes: List[int], dropout_p: float, use_batchnorm: bool) -> str:
    parts = ["hidden=" + "-".join(map(str, hidden_sizes)), "act=ReLU"]
    if dropout_p and dropout_p > 0:
        parts.append(f"dropout={dropout_p}")
    if use_batchnorm:
        parts.append("batchnorm=1")
    return ";".join(parts)


def append_run_row(row: Dict[str, object], csv_path: Path) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        "experiment_id",
        "dataset",
        "seed",
        "model_summary",
        "optimizer",
        "lr",
        "momentum",
        "weight_decay",
        "epochs_trained",
        "best_val_accuracy",
        "best_val_loss",
    ]

    file_exists = csv_path.exists()

    with csv_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists or csv_path.stat().st_size == 0:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in fieldnames})


def plot_curves(history: Dict[str, List[float]], title: str, out_path: Path) -> None:
    epochs = list(range(1, len(history["train_loss"]) + 1))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[0].grid(True)
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="train")
    axes[1].plot(epochs, history["val_acc"], label="val")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].grid(True)
    axes[1].legend()

    fig.suptitle(title)
    fig.tight_layout()

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=160)
    plt.close(fig)


def run_experiment(
    experiment_id: str,
    *,
    hidden_sizes: List[int],
    dropout_p: float,
    use_batchnorm: bool,
    optimizer_name: str,
    lr: float,
    momentum: float = 0.0,
    weight_decay: float = 0.0,
    max_epochs: int = 20,
    early_stopping: Optional[EarlyStopping] = None,
) -> Dict[str, object]:
    set_seed(SEED)

    input_dim = 28 * 28
    num_classes = len(train_full.classes)

    model = MLP(
        input_dim=input_dim,
        num_classes=num_classes,
        hidden_sizes=hidden_sizes,
        dropout_p=dropout_p,
        use_batchnorm=use_batchnorm,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer_name_norm = optimizer_name.lower()
    if optimizer_name_norm == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        opt_label = "Adam"
    elif optimizer_name_norm == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay,
        )
        opt_label = "SGD"
    else:
        raise ValueError("optimizer_name must be 'adam' or 'sgd'")

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        va_loss, va_acc = evaluate(model, val_loader, criterion, DEVICE)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_val_loss = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if early_stopping is not None:
            metric_value = va_acc if early_stopping.mode == "max" else va_loss
            if early_stopping.step(metric_value):
                break

    epochs_trained = len(history["train_loss"])

    result = {
        "experiment_id": experiment_id,
        "dataset": "EMNIST_balanced",
        "seed": SEED,
        "model_summary": model_summary(hidden_sizes, dropout_p, use_batchnorm),
        "optimizer": opt_label,
        "lr": lr,
        "momentum": (momentum if opt_label == "SGD" else 0.0),
        "weight_decay": weight_decay,
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
        "history": history,
        "best_state": best_state,
    }

    append_run_row(
        {k: v for k, v in result.items() if k not in {"history", "best_state"}},
        ARTIFACTS_DIR / "runs.csv",
    )

    return result

In [13]:
# Part A (S08): E1–E4

BASE_HIDDEN = [512, 256, 128]
BASE_LR = 1e-3
BASE_EPOCHS = 20

E1 = run_experiment(
    "E1",
    hidden_sizes=BASE_HIDDEN,
    dropout_p=0.0,
    use_batchnorm=False,
    optimizer_name="adam",
    lr=BASE_LR,
    max_epochs=BASE_EPOCHS,
    early_stopping=None,
)

E2 = run_experiment(
    "E2",
    hidden_sizes=BASE_HIDDEN,
    dropout_p=0.3,
    use_batchnorm=False,
    optimizer_name="adam",
    lr=BASE_LR,
    max_epochs=BASE_EPOCHS,
    early_stopping=None,
)

E3 = run_experiment(
    "E3",
    hidden_sizes=BASE_HIDDEN,
    dropout_p=0.0,
    use_batchnorm=True,
    optimizer_name="adam",
    lr=BASE_LR,
    max_epochs=BASE_EPOCHS,
    early_stopping=None,
)

candidates = [E2, E3]
best_pre = max(candidates, key=lambda r: r["best_val_accuracy"])

print("Best (pre-earlystop) between E2/E3:", best_pre["experiment_id"], "val_acc=", best_pre["best_val_accuracy"])

E4 = run_experiment(
    "E4",
    hidden_sizes=BASE_HIDDEN,
    dropout_p=(0.3 if best_pre["experiment_id"] == "E2" else 0.0),
    use_batchnorm=(best_pre["experiment_id"] == "E3"),
    optimizer_name="adam",
    lr=BASE_LR,
    max_epochs=30,
    early_stopping=EarlyStopping(patience=5, mode="max"),
)

plot_curves(E4["history"], title="E4 (best + EarlyStopping)", out_path=FIGURES_DIR / "curves_best.png")
print("Saved curves_best.png")

best_state = E4["best_state"]
assert best_state is not None

torch.save(best_state, ARTIFACTS_DIR / "best_model.pt")

best_cfg = {
    "dataset": "EMNIST_balanced",
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "val_ratio": VAL_RATIO,
    "hidden_sizes": BASE_HIDDEN,
    "dropout_p": (0.3 if best_pre["experiment_id"] == "E2" else 0.0),
    "use_batchnorm": (best_pre["experiment_id"] == "E3"),
    "optimizer": "Adam",
    "lr": BASE_LR,
    "weight_decay": 0.0,
    "early_stopping": {"patience": 5, "mode": "max"},
}

(ARTIFACTS_DIR / "best_config.json").write_text(json.dumps(best_cfg, indent=2), encoding="utf-8")
print("Saved best_model.pt and best_config.json")

Best (pre-earlystop) between E2/E3: E2 val_acc= 0.8547429078014185
Saved curves_best.png
Saved best_model.pt and best_config.json


In [14]:
# Final test evaluation (ONLY ONCE)

@torch.no_grad()
def evaluate_accuracy_only(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    total_correct = 0
    total = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        total_correct += (torch.argmax(logits, dim=1) == yb).sum().item()
        total += yb.size(0)
    return total_correct / total


NUM_CLASSES = len(train_full.classes)

best_model = MLP(
    input_dim=28 * 28,
    num_classes=NUM_CLASSES,
    hidden_sizes=BASE_HIDDEN,
    dropout_p=best_cfg["dropout_p"],
    use_batchnorm=best_cfg["use_batchnorm"],
).to(DEVICE)

state = torch.load(ARTIFACTS_DIR / "best_model.pt", map_location="cpu")
best_model.load_state_dict(state)

# Ensure eval uses correct device
best_model.to(DEVICE)

test_acc = evaluate_accuracy_only(best_model, test_loader, DEVICE)
print("test_accuracy (best model):", test_acc)

test_accuracy (best model): 0.8562765957446808


In [15]:
# Part B (S09): O1–O3 (fixed architecture = best_cfg)

ARCH_HIDDEN = best_cfg["hidden_sizes"]
ARCH_DROPOUT = best_cfg["dropout_p"]
ARCH_BN = best_cfg["use_batchnorm"]

O1 = run_experiment(
    "O1",
    hidden_sizes=ARCH_HIDDEN,
    dropout_p=ARCH_DROPOUT,
    use_batchnorm=ARCH_BN,
    optimizer_name="adam",
    lr=1e-1,
    max_epochs=8,
)

O2 = run_experiment(
    "O2",
    hidden_sizes=ARCH_HIDDEN,
    dropout_p=ARCH_DROPOUT,
    use_batchnorm=ARCH_BN,
    optimizer_name="adam",
    lr=1e-5,
    max_epochs=8,
)

O3 = run_experiment(
    "O3",
    hidden_sizes=ARCH_HIDDEN,
    dropout_p=ARCH_DROPOUT,
    use_batchnorm=ARCH_BN,
    optimizer_name="sgd",
    lr=5e-3,
    momentum=0.9,
    weight_decay=1e-4,
    max_epochs=15,
)

# Plot LR extremes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs1 = list(range(1, len(O1["history"]["train_loss"]) + 1))
epochs2 = list(range(1, len(O2["history"]["train_loss"]) + 1))

axes[0].plot(epochs1, O1["history"]["train_loss"], label="train")
axes[0].plot(epochs1, O1["history"]["val_loss"], label="val")
axes[0].set_title("O1: lr too big")
axes[0].set_xlabel("epoch")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(epochs2, O2["history"]["train_loss"], label="train")
axes[1].plot(epochs2, O2["history"]["val_loss"], label="val")
axes[1].set_title("O2: lr too small")
axes[1].set_xlabel("epoch")
axes[1].grid(True)
axes[1].legend()

fig.suptitle("LR extremes (loss curves)")
fig.tight_layout()

out_path = FIGURES_DIR / "curves_lr_extremes.png"
fig.savefig(out_path, dpi=160)
plt.close(fig)

print("Saved curves_lr_extremes.png")

Saved curves_lr_extremes.png


In [16]:
import sys

best_A_id = "E4"
best_val_acc = float(E4["best_val_accuracy"])
best_val_loss = float(E4["best_val_loss"])

report_text = f"""# HW08-09 – PyTorch MLP: регуляризация и оптимизация обучения

## 1. Кратко: что сделано

- Датасет: KMNIST (10 классов, изображения 28×28).
- Часть A (S08): E1–E4 (base / Dropout / BatchNorm / EarlyStopping) + выбор лучшей модели по val_accuracy.
- Часть B (S09): O1/O2 для диагностики плохих LR по кривым + O3 (SGD+momentum + weight decay).

## 2. Среда и воспроизводимость

- Python: {sys.version.split()[0]}
- torch / torchvision: {torch.__version__} / {__import__('torchvision').__version__}
- Устройство (CPU/GPU): {DEVICE}
- Seed: {SEED}
- Среда: запуск из venv (используем интерпретатор: `{sys.executable}`)
- Как запустить: открыть `HW08-09.ipynb` и выполнить Run All.

## 3. Данные

- Датасет: KMNIST
- Разделение: train/val/test (val отделён от train: {int((1-VAL_RATIO)*100)}/{int(VAL_RATIO*100)}; test — стандартный torchvision test split)
- Transform: ToTensor()
- Комментарий: изображения маленькие (28×28), классы похожи (символы), поэтому ёмкая MLP может переобучаться без регуляризации.

## 4. Базовая модель и обучение

- Модель MLP: Flatten → Linear/ReLU блоки → logits(10)
- Hidden sizes: {BASE_HIDDEN}
- Loss: CrossEntropyLoss
- Базовый optimizer (часть A): Adam (lr={BASE_LR})
- Batch size: {BATCH_SIZE}
- Epochs (макс): {BASE_EPOCHS} (для E4 максимум 50 с EarlyStopping)
- EarlyStopping: patience=5, mode=max (val_accuracy)

## 5. Часть A (S08): регуляризация (E1-E4)

- E1 (base): без Dropout/BatchNorm
- E2 (Dropout): Dropout(p=0.3)
- E3 (BatchNorm): BatchNorm1d между Linear и ReLU
- E4 (EarlyStopping): выбран лучший из (E2/E3) по val_accuracy + EarlyStopping

## 6. Часть B (S09): LR, оптимизаторы, weight decay (O1-O3)

- O1: Adam, lr=1e-1 (слишком большой)
- O2: Adam, lr=1e-5 (слишком маленький)
- O3: SGD+momentum (momentum=0.9) + weight_decay=1e-4 (lr=5e-3)

## 7. Результаты

Ссылки на файлы:

- Таблица результатов: `./artifacts/runs.csv`
- Лучшая модель: `./artifacts/best_model.pt`
- Конфиг лучшей модели: `./artifacts/best_config.json`
- Кривые лучшего прогона: `./artifacts/figures/curves_best.png`
- Кривые “плохих LR”: `./artifacts/figures/curves_lr_extremes.png`

Короткая сводка:

- Лучший эксперимент части A: {best_A_id}
- Лучшая val_accuracy (E4): {best_val_acc:.4f}
- Лучшая val_loss (E4): {best_val_loss:.4f}
- Итоговая test_accuracy (best model): {test_acc:.4f}
- O1: ожидаемо нестабильный/плохой loss при слишком большом lr (см. `curves_lr_extremes.png`).
- O2: ожидаемо почти нет прогресса при слишком маленьком lr (см. `curves_lr_extremes.png`).
- O3: SGD+momentum+weight_decay даёт сопоставимое обучение при корректном lr; weight decay работает как L2-регуляризация.

## 8. Анализ

- В E1 обычно видно более сильную склонность к переобучению: train метрика улучшается быстрее, чем val.
- Dropout (E2) ограничивает ко-адаптацию нейронов и часто улучшает обобщение ценой замедления обучения.
- BatchNorm (E3) стабилизирует распределения активаций и может ускорять/стабилизировать оптимизацию.
- EarlyStopping (E4) останавливает обучение после плато по val_accuracy и помогает не "перегонять" модель в переобучение.
- O1 "плохой" тем, что шаг оптимизации слишком агрессивный — кривая loss может колебаться или расти.
- O2 "плохой" тем, что шаг слишком мал — loss/accuracy меняются очень медленно.
- SGD+momentum на разумном lr вместе с weight decay может давать более гладкие кривые и служит хорошей базой для тюнинга.

## 9. Итоговый вывод

Для KMNIST разумно брать MLP из {BASE_HIDDEN} с умеренной регуляризацией (Dropout или BatchNorm) и добавлять EarlyStopping, чтобы не переобучаться. Для отладки обучения полезно смотреть на кривые и быстро диагностировать слишком большой/маленький lr. В дальнейшем можно попробовать: (1) другой dropout_p / weight_decay, (2) scheduler для lr.

## 10. Приложение (опционально)

- Дополнительные сравнения (если делались) можно добавить графиками в `./artifacts/figures/`.
"""

with open("rep", "w", encoding="utf-8") as f:
    f.write(report_text)